**Import libraries**

In [12]:
import pandas as pd
import numpy as np
import keras

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Load the dataset

In [13]:
df = pd.read_csv("/content/driving_data.csv")

print(df.head())
print(df.shape)
print(df.isnull().sum())

   distance_front  distance_left  distance_right  distance_back  speed  \
0             168             80              80            168   0.00   
1             168             80              80            168   0.00   
2             168             80              78            168   3.52   
3             168             80              78            168   7.26   
4             168             80              78            168  11.00   

   direction  throttle  steering  
0       90.0         0         0  
1       90.0         1         0  
2       90.0         1         0  
3       90.0         1         0  
4       90.0         1         0  
(8466, 8)
distance_front    0
distance_left     0
distance_right    0
distance_back     0
speed             0
direction         0
throttle          0
steering          0
dtype: int64


Prepare inputs and targets

In [14]:
feature_cols = [
    "distance_front",
    "distance_left",
    "distance_right",
    "distance_back",
    "speed",
    "direction",
]

X = df[feature_cols].to_numpy(dtype=np.float32)

# Throttle: 0 or 1
y_throttle = df["throttle"].to_numpy(dtype=np.float32)

# Steering: convert -1, 0, 1 into 0, 1, 2
y_steering = df["steering"].map({
    -1: 0,
     0: 1,
     1: 2,
}).to_numpy(dtype=np.int32)

print("X:", X.shape)
print("Throttle:", y_throttle.shape)
print("Steering:", y_steering.shape)

X: (8466, 6)
Throttle: (8466,)
Steering: (8466,)


Split the dataset


In [15]:
X_train, X_temp, throttle_train, throttle_temp, steering_train, steering_temp = train_test_split(
    X,
    y_throttle,
    y_steering,
    test_size=0.30,
    random_state=42,
)

X_val, X_test, throttle_val, throttle_test, steering_val, steering_test = train_test_split(
    X_temp,
    throttle_temp,
    steering_temp,
    test_size=0.50,
    random_state=42,
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (5926, 6)
Validation: (1270, 6)
Testing: (1270, 6)


Standardise the inputs

In [16]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

Build the neural network



In [17]:
inputs = keras.Input(shape=(6,))

x = keras.layers.Dense(64, activation="relu")(inputs)
x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Dense(16, activation="relu")(x)

throttle_output = keras.layers.Dense(
    1,
    activation="sigmoid",
    name="throttle",
)(x)

steering_output = keras.layers.Dense(
    3,
    activation="softmax",
    name="steering",
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=[throttle_output, steering_output],
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │        448 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      2,080 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 16)        │        528 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ throttle (Dense)    │ (None, 1)         │         17 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ steering (Dense)    │ (None, 3)         │         51 │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,124 (12.20 KB)

 Trainable params: 3,124 (12.20 KB)

 Non-trainable params: 0 (0.00 B)

Compile the model

In [18]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss={
        "throttle": "binary_crossentropy",
        "steering": "sparse_categorical_crossentropy",
    },
    metrics={
        "throttle": ["accuracy"],
        "steering": ["accuracy"],
    },
)

Train the model

In [19]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
    ),
]

history = model.fit(
    X_train,
    {
        "throttle": throttle_train,
        "steering": steering_train,
    },
    validation_data=(
        X_val,
        {
            "throttle": throttle_val,
            "steering": steering_val,
        },
    ),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.2721 - steering_accuracy: 0.7347 - steering_loss: 0.6934 - throttle_accuracy: 0.7275 - throttle_loss: 0.5758 - val_loss: 1.1583 - val_steering_accuracy: 0.7740 - val_steering_loss: 0.6011 - val_throttle_accuracy: 0.7331 - val_throttle_loss: 0.5566
Epoch 2/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.1477 - steering_accuracy: 0.7626 - steering_loss: 0.6001 - throttle_accuracy: 0.7335 - throttle_loss: 0.5462 - val_loss: 1.1227 - val_steering_accuracy: 0.7764 - val_steering_loss: 0.5839 - val_throttle_accuracy: 0.7386 - val_throttle_loss: 0.5385
Epoch 3/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.1187 - steering_accuracy: 0.7616 - steering_loss: 0.5852 - throttle_accuracy: 0.7386 - throttle_loss: 0.5332 - val_loss: 1.1028 - val_steering_accuracy: 0.7772 - val_steering_loss: 0.5725 - val_throttle_accuracy: 0.7457 - val_throttle_loss: 0.5302
Epoch 4/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.0965

Evaluate the model

In [20]:
results = model.evaluate(
    X_test,
    {
        "throttle": throttle_test,
        "steering": steering_test,
    },
    return_dict=True,
)

print(results)

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7853 - steering_accuracy: 0.8181 - steering_loss: 0.4406 - throttle_accuracy: 0.8512 - throttle_loss: 0.3448
{'loss': 0.7853490114212036, 'steering_accuracy': 0.8181102275848389, 'steering_loss': 0.4406301975250244, 'throttle_accuracy': 0.8511810898780823, 'throttle_loss': 0.34477508068084717}


Check individual steering classes

In [21]:
from sklearn.metrics import classification_report

predictions = model.predict(X_test)

predicted_throttle = (
    predictions[0].ravel() >= 0.5
).astype(int)

predicted_steering = np.argmax(predictions[1], axis=1)

print("Throttle results:")
print(classification_report(
    throttle_test.astype(int),
    predicted_throttle,
    zero_division=0,
))

print("Steering results:")
print(classification_report(
    steering_test,
    predicted_steering,
    labels=[0, 1, 2],
    target_names=["right", "straight", "left"],
    zero_division=0,
))

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Throttle results:
              precision    recall  f1-score   support

           0       0.88      0.91      0.90       918
           1       0.75      0.69      0.72       352

    accuracy                           0.85      1270
   macro avg       0.82      0.80      0.81      1270
weighted avg       0.85      0.85      0.85      1270

Steering results:
              precision    recall  f1-score   support

       right       0.92      0.56      0.69        61
    straight       0.83      0.96      0.89       938
        left       0.72      0.40      0.51       271

    accuracy                           0.82      1270
   macro avg       0.82      0.64      0.70      1270
weighted avg       0.81      0.82      0.80      1270



In [22]:
import joblib

model.save("/content/driving_model.keras")
joblib.dump(scaler, "/content/scaler.pkl")

['/content/scaler.pkl']